In [1]:
!pip install transformers

In [ ]:
from transformers import pipeline

# Create a text-generation pipeline.
# The pipeline handles tokenization, model inference,
# generation, and decoding for us.
generator = pipeline("text-generation")

prompt = "Artificial intelligence is "

# Lower temperature → more predictable output
output = generator(
    prompt,
    max_new_tokens=30,
    temperature=0.1,
    do_sample=True
)
print("Temperature = 0.1")
print(output)

# Medium temperature → more variety
output = generator(
    prompt,
    max_new_tokens=30,
    temperature=0.7,
    do_sample=True
)
print("\nTemperature = 0.7")
print(output)

# Higher temperature → more random/varied output
output = generator(
    prompt,
    max_new_tokens=30,
    temperature=1.5,
    do_sample=True
)
print("\nTemperature = 1.5")
print(output)

In [ ]:
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
import torch

prompt = "AI is really is really good thing"

# Part 1: Using the pipeline

sentiment_pipeline = pipeline("sentiment-analysis")

pipeline_output = sentiment_pipeline(prompt)

print("Pipeline output:")
print(pipeline_output)

# Part 2: Doing the same task manually

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# Load tokenizer and model separately
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# Convert text into model inputs
inputs = tokenizer(
    prompt,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Run the model
outputs = model(**inputs)

# Convert logits into probabilities
probabilities = torch.softmax(outputs.logits, dim=-1)

# Select the class with the highest probability
predicted_class = torch.argmax(
    probabilities,
    dim=-1
).item()

# Convert class ID into a readable label
predicted_label = model.config.id2label[predicted_class]

print("\nManual output:")
print(predicted_label)
print(
    "Probability:",
    probabilities[0][predicted_class].item()
)

In [ ]:
from transformers import pipeline

# Zero-shot classification allows us to provide
# our own candidate labels at inference time.
zero_shot_pipeline = pipeline("zero-shot-classification")

prompt = (
    "You are receiving this email because you joined the "
    "Metro Inc. Talent Community on 18/07/2023. "
    "You will receive these messages every 7 day(s). "
    "Your Job Alert matched the following jobs at careers.metro.ca."
)

result = zero_shot_pipeline(
    prompt,
    candidate_labels=["spam", "not-spam"]
)

print(result)

In [ ]:
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification
)

checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Base Transformer model
base_model = AutoModel.from_pretrained(checkpoint)

# Transformer + sequence classification head
classification_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint
)

prompt = "AI is really great tool"

# Tokenize the input
inputs = tokenizer(
    prompt,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Output from the base Transformer
base_output = base_model(**inputs)

# Output from the model with a classification head
classification_output = classification_model(**inputs)

print("Base Transformer output shape:")
print(base_output.last_hidden_state.shape)

print("\nClassification model output shape:")
print(classification_output.logits.shape)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

checkpoint = "gpt2"

# Load tokenizer and causal language model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)

prompt = "AI is really"

# Convert text into token IDs
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Inference only — we do not need gradients
with torch.no_grad():
    logits = model(**inputs).logits

# Get the logits corresponding to the next-token prediction
next_token_logits = logits[0, -1, :]

# Convert logits into probabilities
probabilities = torch.softmax(
    next_token_logits,
    dim=-1
)

# Find the 5 tokens with the highest probabilities
top5 = torch.topk(
    probabilities,
    5
)

# Display the token and its probability
for score, token_id in zip(top5.values, top5.indices):
    print(
        tokenizer.decode(token_id),
        score.item()
    )

In [6]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

sentences = [
    "AI is really is really good thing",
    "I love AI"
]

# Tokenize multiple sentences with padding
inputs = tokenizer(
    sentences,
    padding=True,
    return_tensors="pt"
)

print("Input IDs:")
print(inputs["input_ids"])

print("\nAttention mask:")
print(inputs["attention_mask"])

# Decode each sentence separately
print("\nDecoded sentences:")

for token_ids in inputs["input_ids"]:
    print(tokenizer.decode(token_ids))

['[CLS] ai is really is really good thing [SEP]', '[CLS] i love ai [SEP] [PAD] [PAD] [PAD] [PAD]']
